# Part 3

The goal in Part 3 is to bring a modern pretrained vision-language transformer to the same captioning task. We will fine-tune `microsoft/git-base` (Generative Image-to-Text Transformer; Wang et al., 2022) on Flickr8k and compute the same corpus-level BLEU metrics as Parts 1 and 2, so all three models are directly comparable on the same test split.

Why this model?
- In Exercise Set 17 we learned about transfer learning: take a network pretrained on a huge dataset (ResNet-50 on ImageNet), freeze its parameters, and reuse the visual features it already learned. In Part 2, we applied that recipe — a frozen pretrained encoder feeding a decoder trained from scratch. In Part 3, we'll take transfer learning one step further: here the encoder *and* the decoder will be pretrained together on millions of image-caption pairs, and instead of freezing anything, we will *fine-tune* every parameter with a small learning rate so the model can adapt to Flickr8k without forgetting what it already knows.
- The transformer replaces the LSTM's recurrence with self-attention. In Part 2 the entire image had to squeeze through a single 2,048-dimensional vector that initialized the LSTM exactly once, and information about earlier words had to survive a chain of step-by-step hidden-state updates. In a transformer, every generation step attends directly to every image patch and every previous word — there is no bottleneck and no vanishing gradient across time steps, because any two positions are connected in one step rather than through a chain of multiplications.
- `microsoft/git-base` is the model used in HuggingFace's official image-captioning tutorial, so it is well-documented and a realistic representative of the current standard approach.

What we did:
1. Same `75% train / 12.5% val / 12.5% test` split as Parts 1 and 2.
2. Load the `microsoft/git-base` processor and model from HuggingFace. The processor handles image preprocessing and tokenization jointly in one call.
3. Build a PyTorch `Dataset` that returns one processor-encoded sample per image-caption pair (5 per image, same as Part 2), with padding positions masked out of the loss via the `-100` label convention.
4. Fine-tune the full model with AdamW (`lr = 5e-5`) and early stopping on validation loss. Cache trained weights so the cell is skipped on re-run.
5. Generate captions for the test set with greedy decoding via `model.generate()`.
6. Compute corpus-level BLEU-1 through BLEU-4 using the same setup as Parts 1 and 2.
7. Visualize a sample of test image predictions alongside their reference captions.

Import all the libraries and tools needed below.

In [3]:
# import sys
# ! pip install torch transformers
import os
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

import re
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForCausalLM
import nltk

nltk.download('punkt', quiet = True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

Set the paths to the dataset. The `archive/` folder is expected to sit alongside this notebook. Also pick the compute device — PyTorch (unlike keras) requires moving the model and every batch to the device explicitly. And we're using PyTorch because `microsoft/git-base` is loaded via HuggingFace's transformers library, and its model classes are built on built on PyTorch.

In [4]:
ARCHIVE_DIR  = os.path.join(os.getcwd(), 'archive')
CAPTIONS_CSV = os.path.join(ARCHIVE_DIR, 'captions.txt')
IMAGES_DIR   = os.path.join(ARCHIVE_DIR, 'Images')
CACHE_DIR    = os.path.join(os.getcwd(), 'cache')

os.makedirs(CACHE_DIR, exist_ok = True)

# use the GPU if available — CUDA on NVIDIA, MPS on Apple Silicon, otherwise CPU
DEVICE = (
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print('Device :', DEVICE)

Device : mps


### 1)

Load the captions file and reproduce the same train/val/test split as Parts 1 and 2 (75% / 12.5% / 12.5%, seed 42). Splitting at the image level ensures all five captions for a given image stay in the same partition — no caption leaks across splits.

The validation set is used here (as in Part 2) because fine-tuning requires monitoring held-out loss to decide when to stop.

In [5]:
df = pd.read_csv(CAPTIONS_CSV)
df.columns = df.columns.str.strip()   # strip white space in front of 'image'

# tokenize and lowercase — same helper as Parts 1 and 2
def tokenize(text):
    return re.findall(r'[a-z]+', text.lower())

df['tokens'] = df['caption'].apply(tokenize)

# ensure reproducibility
all_images = sorted(df['image'].unique())
random.seed(42)
random.shuffle(all_images)

# train/val/test split
n       = len(all_images)
n_train = int(0.75  * n)
n_val   = int(0.125 * n)

train_imgs = set(all_images[:n_train])
val_imgs   = set(all_images[n_train : n_train + n_val])
test_imgs  = set(all_images[n_train + n_val:])

train_list = sorted(train_imgs)
val_list   = sorted(val_imgs)
test_list  = sorted(test_imgs)

print('Train images :', len(train_list))
print('Val images   :', len(val_list))
print('Test images  :', len(test_list))

Train images : 6068
Val images   : 1011
Test images  : 1012


### 2)

Load the `microsoft/git-base` processor and model. 

GIT was pretrained on hundreds of millions of image-text pairs using a standard next-token prediction objective over the text tokens conditioned on the image tokens. Fine-tuning on Flickr8k specializes this prior to the dataset's caption style and vocabulary.

This is the model used in the official HuggingFace image-captioning tutorial.

Reasoning:
- https://arxiv.org/abs/2205.14100 (GIT paper)
- https://arxiv.org/abs/2010.11929 (ViT paper)
- https://huggingface.co/docs/transformers/main/en/tasks/image_captioning (the HuggingFace image-captioning tutorial this part follows)

In [6]:
MODEL_NAME = 'microsoft/git-base'

# load the processor class associated with the model (above) we are going to fine-tune
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model                : {MODEL_NAME}')
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

Loading weights: 100%|██████████| 305/305 [00:00<00:00, 10592.90it/s]


Model                : microsoft/git-base
Total parameters     : 176,619,066
Trainable parameters : 176,619,066


### 3)

Define a custom PyTorch `Dataset` where each of the 5 captions per image is treated as a separate (image, caption) sample, then wrap the train and val sets in `DataLoader`s for mini-batching.

A single processor call with both `images` and `text` returns three aligned tensors: `pixel_values` (preprocessed image), `input_ids` (tokenized caption), and `attention_mask` (1 at real tokens, 0 at padding). The `labels` tensor is then constructed manually by copying `input_ids` and replacing every padding position — identified via `attention_mask == 0` — with `-100`, so those slots are ignored by the cross-entropy loss.

The `DataLoader`s use a batch size of 8 (small because all 129M parameters are being updated), shuffle training samples each epoch to avoid ordering effects, and set `num_workers = 0` to avoid multiprocessing conflicts on macOS.

Reasoning:
- https://huggingface.co/docs/transformers/main/en/model_doc/git
- https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

In [7]:
MAX_LENGTH = 50   # the longest Flickr8k caption is 36 words — 50 subword tokens covers it plus specials

class Flickr8kGITDataset(Dataset):
    # one sample per image-caption pair — 5 samples per image, same as Part 2
    def __init__(self, img_list, df, processor, images_dir, max_length):
        self.processor  = processor
        self.images_dir = images_dir
        self.max_length = max_length
        # flatten each image's five captions into separate (filename, caption) pairs
        self.samples = [
            (fname, caption)
            for fname in img_list
            for caption in df[df['image'] == fname]['caption']
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        fname, caption = self.samples[i]
        image = Image.open(os.path.join(self.images_dir, fname)).convert('RGB') # opens image

        # joint call returns pixel_values, input_ids and attention_mask, already aligned
        # (the GIT tokenizer is uncased, so no manual lowercasing is needed)
        encoding = self.processor(
            images         = image,
            text           = caption,
            padding        = 'max_length',
            truncation     = True,
            max_length     = self.max_length,
            return_tensors = 'pt'
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()} # drop the batch dimension
                                                                    # strip away redundant single dimension

        # loss target: copy of input_ids with padding positions replaced by -100 (ignored by the loss)
        labels = encoding['input_ids'].clone()
        labels[encoding['attention_mask'] == 0] = -100 # the flag for ignore index
        encoding['labels'] = labels
        return encoding

In [8]:
BATCH_SIZE = 8   # small because the full model is trained

train_dataset = Flickr8kGITDataset(train_list, df, processor, IMAGES_DIR, MAX_LENGTH)
val_dataset = Flickr8kGITDataset(val_list, df, processor, IMAGES_DIR, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True, # reshuffle pairs every epoch
    num_workers = 0
)
val_loader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 0
)

print('Training samples   :', len(train_dataset))
print('Validation samples :', len(val_dataset))
print('Batches per epoch  :', len(train_loader))

Training samples   : 30340
Validation samples : 5055
Batches per epoch  : 3793


### 4)

Fine-tune `microsoft/git-base` on the training dataset. If a cached weights file already exists from a previous run, it is loaded directly and training is skipped — like what we did with caching in Part 2.

**Optimizer** — AdamW (`lr = 5e-5`, `weight_decay = 0.01`). The small learning rate nudges the pretrained parameters toward Flickr8k's caption style without overwriting them (catastrophic forgetting). AdamW applies weight decay directly to the weights rather than through the gradient update, which makes the regularization behave correctly under Adam's per-parameter learning rates.

**Loss** — The model computes its own loss when `labels` are provided: cross-entropy between each position's predicted next-token distribution and the actual next token, skipping the -100 positions — functionally the same masked objective we wrote by hand in Part 2.

**Early stopping** — `EPOCHS = 5`, `PATIENCE = 2`: checkpoints the best val-loss epoch and stops if val loss fails to improve twice in a row, then restores the best weights.

Reasoning:
- https://huggingface.co/docs/transformers/main/en/tasks/image_captioning

In [ ]:
WEIGHTS_PATH = os.path.join(CACHE_DIR, 'git_base_weights.pt')
HISTORY_PATH = os.path.join(CACHE_DIR, 'git_base_history.pkl')

EPOCHS = 5
PATIENCE = 2
LR = 5e-5

def run_epoch(loader, train):
    # one full pass over loader; updates parameters only when train = True
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    for b, batch in enumerate(loader):
        # loops through every variable inside structured dictionary (pixel_values, input_ids, attention_mask, labels)
        #   and ships them into device where the cross-entropy adjustments are computed natively
        batch = {k: v.to(DEVICE) for k, v in batch.items()} # move tensors to the device
        with torch.set_grad_enabled(train):
            outputs = model(**batch) # forward pass — loss is computed internally from labels
            loss = outputs.loss # the model computes cross-entropy adjustments natively
        if train: # gradient descent sequence:
            optimizer.zero_grad() # clear gradients from the previous step
            loss.backward() # backpropagate
            optimizer.step() # update parameters (moves model weights by the miniscule LR to decrease model error)
            if (b + 1) % 500 == 0:
                print(f'  batch {b + 1} / {len(loader)}')
        total_loss += loss.item()
    return total_loss / len(loader)

# check for local cache and load if there
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location = DEVICE))
    history = pickle.load(open(HISTORY_PATH, 'rb'))
    print('Loaded weights from cache — skipping training.')
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr = LR, weight_decay = 0.01)
    history = {'train_loss': [], 'val_loss': []}

    best_val = float('inf')
    patience_left = PATIENCE

    for epoch in range(EPOCHS):
        train_loss = run_epoch(train_loader, train = True)
        val_loss = run_epoch(val_loader, train = False)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch + 1}/{EPOCHS} — train loss: {train_loss:.4f} — val loss: {val_loss:.4f}')

        # validation checpoint and early stopping
        if val_loss < best_val:
            best_val = val_loss
            patience_left = PATIENCE
            torch.save(model.state_dict(), WEIGHTS_PATH) # checkpoint the best epoch so far
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f'Early stopping — no val improvement for {PATIENCE} epochs.')
                break

    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location = DEVICE)) # restore the best checkpoint
    pickle.dump(history, open(HISTORY_PATH, 'wb'))
    print('Training complete. Weights saved to cache.')

# plot training curve (works from cache too, since the history is saved alongside the weights)
plt.figure(figsize = (8, 4))
plt.plot(history['train_loss'], label = 'Train loss')
plt.plot(history['val_loss'], label = 'Val loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('GIT — Training and Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

  batch 500 / 3793
  batch 1000 / 3793


### 5)

Generate a caption for every test image using `model.generate()`, passing only `pixel_values` — no ground-truth text. Generation starts from beginning of sequence (BOS) and proceeds one token at a time, feeding each prediction back in, until the model emits until end of sequence (EOS) or `max_new_tokens`.

`generate()` uses greedy decoding (argmax at each step) by default, matching Part 2's inference strategy so the decoding method is held constant across models. The key difference is that here every step's prediction attends to all 196 visual tokens, rather than relying on a hidden state initialized once from a single image vector.

Reasoning:
- Batches of 32 keep the device busy without exhausting memory (no gradients are stored at inference, so inference batches can be larger than training batches).
- `skip_special_tokens=True` does the equivalent of Part 2 stripping `<START>`, `<END>`, and `<PAD>`.
- https://huggingface.co/blog/how-to-generate

In [ ]:
def generate_captions(img_list, batch_size = 32, max_new_tokens = 40):
    # greedy-decode captions for a list of image filenames
    model.eval()
    all_captions = []

    for start in range(0, len(img_list), batch_size):
        batch_fnames = img_list[start : start + batch_size]
        images = [
            Image.open(os.path.join(IMAGES_DIR, fname)).convert('RGB')
            for fname in batch_fnames
        ]
        pixel_values = processor(images = images, return_tensors = 'pt').pixel_values.to(DEVICE)

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values   = pixel_values,
                max_new_tokens = max_new_tokens
            )
        all_captions += processor.batch_decode(generated_ids, skip_special_tokens = True)

        if (start // batch_size + 1) % 10 == 0: # print progress
            print(f'  {min(start + batch_size, len(img_list))} / {len(img_list)} done')

    return all_captions

generated_strings = generate_captions(test_list)

print('Inference complete.')
print('Example generated caption :', generated_strings[0])

### 6)

Compute corpus-level BLEU-1 through BLEU-4 on the test set using the same setup as Parts 1 and 2 so the scores are directly comparable.

Each test image has five reference captions. BLEU-1 measures unigram overlap; BLEU-4 requires matching 4-gram sequences and is therefore much stricter. The same `SmoothingFunction().method1` is applied to BLEU-2 through BLEU-4 to handle cases where a higher-order n-gram precision is zero.

Each generated string is passed through the same `tokenize()` helper as the references (lowercase, strip punctuation), so evaluation is identical across all three parts — any change in score reflects the model architecture, not a different evaluation setup.

Reasoning:
- Computing the same BLEU metrics against the same test split as Part 1 and 2 isolates the effect of the model architecture. Any change in score reflects the generative decoder, not a different evaluation setup.
- https://www.nltk.org/api/nltk.translate.bleu_score.html

In [ ]:
# map each test image name to its five reference captions (same as Parts 1 and 2)
test_captions = (
    df[df['image'].isin(test_imgs)]
    .groupby('image')['caption']
    .apply(list)
    .to_dict()
)

# build references and hypotheses in the format corpus_bleu expects
references = [] # list of lists-of-lists (5 tokenized refs per image)
hypotheses = [] # list of tokenized predictions

for i, fname in enumerate(test_list):
    refs = [tokenize(c) for c in test_captions[fname]] # tokenize each test image's 5 reference captions
    hyp  = tokenize(generated_strings[i])               # tokenize the generated caption
    references.append(refs)
    hypotheses.append(hyp)

print('Example prediction    :', ' '.join(hypotheses[0]))
print('Example reference [0] :', ' '.join(references[0][0]))

In [ ]:
smoother = SmoothingFunction().method1 # add epsilon counts to precision with 0 counts

bleu1 = corpus_bleu(references, hypotheses, weights = (1, 0, 0, 0))
bleu2 = corpus_bleu(references, hypotheses, weights = (0.5, 0.5, 0, 0),         smoothing_function = smoother)
bleu3 = corpus_bleu(references, hypotheses, weights = (1/3, 1/3, 1/3, 0),       smoothing_function = smoother)
bleu4 = corpus_bleu(references, hypotheses, weights = (0.25, 0.25, 0.25, 0.25), smoothing_function = smoother)

print(f'BLEU-1 : {bleu1:.4f}')
print(f'BLEU-2 : {bleu2:.4f}')
print(f'BLEU-3 : {bleu3:.4f}')
print(f'BLEU-4 : {bleu4:.4f}')

The fine-tuned GIT is expected to outperform both the retrieval baseline (Part 1) and the CNN + LSTM encoder-decoder (Part 2), with the gap widening on BLEU-3 and BLEU-4, for two reasons. First, the transformer decoder attends to all 196 image patch tokens at every generation step rather than relying on a single 2,048-dim bottleneck, so it can keep referring back to the image while composing longer phrases. Second, large-scale pretraining means the model is already fluent when it begins fine-tuning — it only has to adapt to Flickr8k's caption style, while Part 2's decoder had to learn English from 30,340 captions alone. The actual numbers from all three parts are tabulated and discussed in the report.

### 7)

Visualize a sample of test images alongside their generated caption and one reference caption. This provides a qualitative check on where the model succeeds and where it fails — complementing the aggregate BLEU scores with concrete examples. The same seed is used as in Part 2, so the same test images are shown for a side-by-side comparison across models.

In [ ]:
n_show   = 4
random.seed(42)
sample_i = random.sample(range(len(test_list)), n_show) # get n_show number of randomly sampled indices to extract images to plot

fig, axes = plt.subplots(n_show, 1, figsize = (10, 4 * n_show))

for row, i in enumerate(sample_i):
    fname = test_list[i]
    pred_text = ' '.join(hypotheses[i]) if hypotheses[i] else '(empty)' # guards against if generate_captions_batch returns an empty list
    ref_text = ' '.join(references[i][0])

    axes[row].imshow(mpimg.imread(os.path.join(IMAGES_DIR, fname)))
    axes[row].axis('off')
    axes[row].set_title(
        f'Prediction — {pred_text}\nReference — {ref_text}',
        fontsize = 8,
        loc = 'left'
    )

plt.suptitle('GIT Fine-tuned on Flickr8k — Sample Test Predictions\n', fontsize = 12)
plt.tight_layout()
plt.show()

### 8) *(Optional)* Comparison — `nlpconnect/vit-gpt2-image-captioning`

`nlpconnect/vit-gpt2-image-captioning` pairs the same kind of ViT image encoder with a GPT-2 language model decoder, glued together by HuggingFace's `VisionEncoderDecoderModel`. It comes already fine-tuned for captioning (on MS COCO), so we run it zero-shot — no Flickr8k fine-tuning — as a second pretrained-transformer reference point.

Two contrasts with GIT worth noting:
- **Architecture** — GIT is a single decoder that reads the visual tokens as a prefix of its own input sequence; ViT-GPT2 keeps the encoder and decoder separate, with the decoder querying the encoder's output through dedicated cross-attention layers. It also uses its own `ViTImageProcessor` and `AutoTokenizer` rather than one unified processor.
- **Training** — zero-shot means it was tuned on COCO's caption style, not Flickr8k's, so it may phrase things differently from the references even when it reads the image correctly — and BLEU, being an n-gram overlap metric, will penalize that. The comparison therefore measures fine-tuning's value, not just architecture.

Reasoning:
- https://huggingface.co/nlpconnect/vit-gpt2-image-captioning
- https://huggingface.co/docs/transformers/main/en/model_doc/vision-encoder-decoder

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

VG_NAME      = 'nlpconnect/vit-gpt2-image-captioning'
vg_processor = ViTImageProcessor.from_pretrained(VG_NAME)
vg_tokenizer = AutoTokenizer.from_pretrained(VG_NAME)
vg_model     = VisionEncoderDecoderModel.from_pretrained(VG_NAME).to(DEVICE)
vg_model.eval()

print('Model loaded :', VG_NAME)

In [ ]:
def generate_vg_captions(img_list, batch_size = 32, max_new_tokens = 40):
    # zero-shot greedy decoding with vit-gpt2 — same loop shape as generate_captions above
    all_captions = []

    for start in range(0, len(img_list), batch_size):
        batch_fnames = img_list[start : start + batch_size]
        images = [
            Image.open(os.path.join(IMAGES_DIR, fname)).convert('RGB')
            for fname in batch_fnames
        ]
        pixel_values = vg_processor(images = images, return_tensors = 'pt').pixel_values.to(DEVICE)

        with torch.no_grad():
            generated_ids = vg_model.generate(pixel_values, max_new_tokens = max_new_tokens)
        all_captions += vg_tokenizer.batch_decode(generated_ids, skip_special_tokens = True)

        if (start // batch_size + 1) % 10 == 0: # print progress
            print(f'  {min(start + batch_size, len(img_list))} / {len(img_list)} done')

    return all_captions

vg_strings    = generate_vg_captions(test_list)
vg_hypotheses = [tokenize(s) for s in vg_strings]

vg_bleu1 = corpus_bleu(references, vg_hypotheses, weights = (1, 0, 0, 0))
vg_bleu2 = corpus_bleu(references, vg_hypotheses, weights = (0.5, 0.5, 0, 0),         smoothing_function = smoother)
vg_bleu3 = corpus_bleu(references, vg_hypotheses, weights = (1/3, 1/3, 1/3, 0),       smoothing_function = smoother)
vg_bleu4 = corpus_bleu(references, vg_hypotheses, weights = (0.25, 0.25, 0.25, 0.25), smoothing_function = smoother)

print('Corpus BLEU — GIT (fine-tuned) vs ViT-GPT2 (zero-shot)')
print(f'BLEU-1 : {bleu1:.4f} vs {vg_bleu1:.4f}')
print(f'BLEU-2 : {bleu2:.4f} vs {vg_bleu2:.4f}')
print(f'BLEU-3 : {bleu3:.4f} vs {vg_bleu3:.4f}')
print(f'BLEU-4 : {bleu4:.4f} vs {vg_bleu4:.4f}')